# 04 — Modelado

**TFM RunnAing** | Fase 4

**Objetivos:**
- 4.1 Split 70/15/15 estratificado por usuario (sin leakage)
- 4.2 Entrenar 5 modelos de ML
- 4.3 Optuna 100 trials (solo XGBoost y LightGBM) — MAE en validación

**Modelos:**
1. Regresión Lineal (baseline)
2. Random Forest
3. Gradient Boosting (sklearn, modelo adicional justificado)
4. XGBoost ← Optuna 100 trials
5. LightGBM ← Optuna 100 trials

**Input** : `data/processed/sessions_features.parquet`  
**Outputs**: `models/` (modelos pkl) + `models/optuna_studies/` (JSON + study pkl)

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.models import get_model, MODEL_REGISTRY
from src.splits import group_train_val_test_split, verify_no_user_overlap
from src.tuning import tune_model

np.random.seed(42)

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
STUDY_DIR = Path('../models/optuna_studies')
STUDY_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK')

## 1. Carga del dataset procesado

In [ ]:
df = pd.read_parquet('../data/processed/sessions_features.parquet')
print(f'Dataset cargado: {df.shape}')
print(f'Columnas: {list(df.columns)}')

FEATURE_COLS = [c for c in df.columns if c not in ('userId', 'gender', 'date', 'trimp')]
print(f'\nFeatures ({len(FEATURE_COLS)}): {FEATURE_COLS}')

X = df[FEATURE_COLS]
y = df['trimp']
groups = df['userId']

print(f'\nX shape : {X.shape}')
print(f'y stats : mean={y.mean():.2f}, std={y.std():.2f}, min={y.min():.2f}, max={y.max():.2f}')
print(f'Usuarios: {groups.nunique():,}')

## 2. Split 70 / 15 / 15 por usuario (Fase 4.1)

Cada usuario completo se asigna a una sola partición → sin leakage entre usuarios.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, g_train, g_val, g_test = \
    group_train_val_test_split(X, y, groups, train_frac=0.70, val_frac=0.15, seed=42)

# Verificar sin leakage
verify_no_user_overlap(g_train, g_val, g_test)

print('=== SPLIT 70/15/15 ===')
print(f'Train  : {len(X_train):>7,} sesiones | {g_train.nunique():>5,} usuarios ({100*len(X_train)/len(X):.1f}%)')
print(f'Val    : {len(X_val):>7,} sesiones | {g_val.nunique():>5,} usuarios ({100*len(X_val)/len(X):.1f}%)')
print(f'Test   : {len(X_test):>7,} sesiones | {g_test.nunique():>5,} usuarios ({100*len(X_test)/len(X):.1f}%)')
print('Leakage check: OK — ningún usuario en más de una partición')

## 3. Modelos 1-3: sin Optuna (Fase 4.2)

Regresión Lineal (baseline), Random Forest y Gradient Boosting con hiperparámetros por defecto.

In [ ]:
trained_models = {}
val_metrics = {}

def eval_metrics(model, X_ev, y_ev, name=''):
    """Calcula MAE, RMSE y R² sobre un conjunto."""
    preds = model.predict(X_ev)
    mae = mean_absolute_error(y_ev, preds)
    rmse = np.sqrt(mean_squared_error(y_ev, preds))
    r2 = r2_score(y_ev, preds)
    if name:
        print(f'  {name:<20} MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

# Regresión Lineal
print('Entrenando Regresión Lineal (baseline)...')
lr = get_model('LinearRegression')
lr.fit(X_train, y_train)
trained_models['LinearRegression'] = lr
val_metrics['LinearRegression'] = eval_metrics(lr, X_val, y_val, 'LinearRegression')

# Random Forest
print('Entrenando Random Forest...')
rf = get_model('RandomForest', n_estimators=300, max_depth=10)
rf.fit(X_train, y_train)
trained_models['RandomForest'] = rf
val_metrics['RandomForest'] = eval_metrics(rf, X_val, y_val, 'RandomForest')

# Gradient Boosting (sklearn)
print('Entrenando Gradient Boosting...')
gb = get_model('GradientBoosting', n_estimators=300, learning_rate=0.05, max_depth=5)
gb.fit(X_train, y_train)
trained_models['GradientBoosting'] = gb
val_metrics['GradientBoosting'] = eval_metrics(gb, X_val, y_val, 'GradientBoosting')

print('\nModelos base entrenados.')

## 4. Optuna 100 trials — XGBoost y LightGBM (Fase 4.3)

- Métrica de optimización: **MAE en validación** (minimizar)
- 100 trials, TPE sampler, seed=42
- Studies guardados en `models/optuna_studies/`
- Mejores hiperparámetros en JSON

⚠️ Esta celda puede tardar 1–3 horas en CPU. Reducir `n_trials` para pruebas rápidas.

In [ ]:
N_TRIALS = 100

print(f'=== XGBoost — Optuna {N_TRIALS} trials ===')
xgb_result = tune_model(
    'XGBoost', X_train, y_train, X_val, y_val,
    n_trials=N_TRIALS, seed=42, study_dir=str(STUDY_DIR)
)
print(f'XGBoost mejor MAE (val): {xgb_result["best_mae"]:.4f}')
print(f'Mejores params         : {xgb_result["best_params"]}')

# Reentrenar con mejores hiperparámetros
xgb_best = get_model('XGBoost', **xgb_result['best_params'])
xgb_best.fit(X_train, y_train)
trained_models['XGBoost'] = xgb_best
val_metrics['XGBoost'] = eval_metrics(xgb_best, X_val, y_val, 'XGBoost')

In [ ]:
print(f'=== LightGBM — Optuna {N_TRIALS} trials ===')
lgb_result = tune_model(
    'LightGBM', X_train, y_train, X_val, y_val,
    n_trials=N_TRIALS, seed=42, study_dir=str(STUDY_DIR)
)
print(f'LightGBM mejor MAE (val): {lgb_result["best_mae"]:.4f}')
print(f'Mejores params          : {lgb_result["best_params"]}')

lgb_best = get_model('LightGBM', **lgb_result['best_params'])
lgb_best.fit(X_train, y_train)
trained_models['LightGBM'] = lgb_best
val_metrics['LightGBM'] = eval_metrics(lgb_best, X_val, y_val, 'LightGBM')

## 5. Comparación en validación — selección del mejor modelo

In [ ]:
val_df = pd.DataFrame(val_metrics).T.reset_index().rename(columns={'index': 'modelo'})
val_df = val_df.sort_values('mae')
print('=== MÉTRICAS EN VALIDACIÓN ===')
print(val_df.to_string(index=False))

best_model_name = val_df.iloc[0]['modelo']
print(f'\nMejor modelo (menor MAE val): {best_model_name}')

## 6. Guardado de modelos y split

In [ ]:
# Guardar todos los modelos entrenados
for name, model in trained_models.items():
    path = MODELS_DIR / f'{name.lower()}_model.pkl'
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'Guardado: {path}')

# Guardar el mejor modelo con alias
best_path = MODELS_DIR / 'best_model.pkl'
with open(best_path, 'wb') as f:
    pickle.dump(trained_models[best_model_name], f)
print(f'Guardado: {best_path}  ({best_model_name})')

# Guardar metadatos del split y mejor modelo para notebooks siguientes
split_meta = {
    'best_model_name': best_model_name,
    'feature_cols': FEATURE_COLS,
    'split': {
        'train_users': g_train.nunique(), 'train_sessions': len(X_train),
        'val_users': g_val.nunique(), 'val_sessions': len(X_val),
        'test_users': g_test.nunique(), 'test_sessions': len(X_test),
    }
}
with open(MODELS_DIR / 'split_meta.json', 'w') as f:
    json.dump(split_meta, f, indent=2)
print(f'Guardado: {MODELS_DIR}/split_meta.json')

# Guardar índices del test para evaluación reproducible
test_index_df = pd.DataFrame({
    'idx': X_test.index,
    'userId': g_test.values,
    'trimp_real': y_test.values,
})
test_index_df.to_parquet(MODELS_DIR / 'test_split.parquet', index=False)
print(f'Guardado: {MODELS_DIR}/test_split.parquet')

In [ ]:
print('=== RESUMEN FASE 4 ===')
print(f'Split: train={len(X_train):,} | val={len(X_val):,} | test={len(X_test):,}')
print(f'Modelos entrenados: {list(trained_models.keys())}')
print(f'Optuna: XGBoost y LightGBM — {N_TRIALS} trials c/u')
print(f'Mejor modelo (val MAE): {best_model_name}')
print(f'\nPróximo paso: notebook 05_evaluation.ipynb')